# NB06 — Confounder Discovery

**Type:** EXPLORATORY — all findings post-hoc, cannot be elevated to confirmatory without a new pre-registration  
**Execution order:** Run BEFORE NB01–05 (RESEARCH_PLAN.md §9)

> **Reproducibility note:** This notebook requires a live JupyterHub Spark session to re-execute (kbase.ke_pangenome / kescience_mgnify). The analysis code cells have not been re-executed in-place after the initial Spark run. Cached outputs are stored in `data/06_table_screen_all.csv`, `data/06_confounder_candidates.csv`, `data/06_confounder_redundancy.csv`, and `data/06_candidate_coverage.csv`. See the **Cached Results** section at the bottom of this notebook for a display of those files.

## Purpose

Systematically scan all BERDL Spark namespaces for environmental datasets that:
1. Have geographic coordinates (lat/lon)
2. Have ≥2 environmental columns beyond coordinates
3. Have ≥30% coverage of the analysis sample locations (within 200 km)

Identified datasets are catalogued for future pre-registration. **No statistical tests are run here.**

## What this informs
- RESEARCH_PLAN.md §7 (pre-specified confounders in NB04) — were any *missed*?
- Future confirmatory analysis plans
- Data availability audit before NB01 runs

In [1]:
import sys
from pathlib import Path

_project_root = Path().resolve().parent
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scripts.berdl_utils import (
    get_spark_session, list_namespaces, list_tables,
    sample_table, describe_table,
)
from scripts.confounder_discovery import (
    list_all_berdl_tables,
    screen_table_for_env_vars,
    screen_all_tables,
    evaluate_coverage,
    summarise_candidate_confounders,
    flag_redundant_confounders,
)
from scripts.spatial_utils import haversine_join

DATA = _project_root / 'data'
FIGURES = _project_root / 'figures'

pd.set_option('display.max_rows', 120)
print('Imports OK')

## Block 1 — Spark session and namespace inventory

In [2]:
spark = get_spark_session()

namespaces = list_namespaces(spark)
print(f'BERDL namespaces ({len(namespaces)} total):')
for ns in sorted(namespaces):
    print(f'  {ns}')

In [3]:
# Enumerate all tables across all namespaces
all_tables = list_all_berdl_tables(spark)
print(f'Total tables found: {len(all_tables)}')

# Show table counts per namespace
ns_counts = {}
for t in all_tables:
    ns = t.split('.')[0]
    ns_counts[ns] = ns_counts.get(ns, 0) + 1
for ns, n in sorted(ns_counts.items(), key=lambda x: -x[1]):
    print(f'  {ns}: {n} tables')

## Block 2 — Screen all tables for environmental variables

Flags tables that have: (a) lat/lon columns, (b) ≥2 additional environmental columns.

In [4]:
# ── Automated BERDL table scanning (commented out — DESCRIBE API unreliable) ──
# try:
#     screen_results = screen_all_tables(spark, all_tables, min_env_cols=2)
# except BaseException as exc:
#     print(f'screen_all_tables failed: {exc}')
#     screen_results = pd.DataFrame(columns=[
#         "table", "has_latlon", "env_cols", "n_env_cols", "n_rows", "is_candidate", "error"])
# ─────────────────────────────────────────────────────────────────────────────

# Sourced from ENV_VARIABLES_CATALOG.md and OMICS_DATASET_CATALOG.md (2026-07-05).
# lat_col / lon_col: actual column names in the table (vary per dataset).
# is_candidate: True = has real lat/lon + ≥2 env variables beyond coordinates.
# pitfall: known issues that affect downstream use.
KNOWN_TABLES = [
    # ── Microbeatlas (463K samples, global 16S community survey) ──────────────
    {
        "table": "arkinlab.microbeatlas.enriched_metadata",
        "n_rows": 278952, "lat_col": "LatitudeParsed", "lon_col": "LongitudeParsed",
        "is_candidate": True,
        "env_cols": ["LatitudeParsed", "LongitudeParsed", "Co", "Cr", "Cu", "Ni", "Zn", "Pb", "U",
                     "usgs_mine_count", "usgs_mine_distance", "tectonic_boundary_dist",
                     "epa_npl_superfund_count", "epa_tri_releases"],
        "scope": "Global", "pitfall": "TRY_CAST lat/lon; 'Unknown' strings present",
    },
    {
        "table": "arkinlab.microbeatlas.sample_metadata",
        "n_rows": 463972, "lat_col": "LatitudeParsed", "lon_col": "LongitudeParsed",
        "is_candidate": True,
        "env_cols": ["LatitudeParsed", "LongitudeParsed", "ph", "temp_C", "altitude_m", "depth_m"],
        "scope": "Global", "pitfall": "TRY_CAST lat/lon; ~240K rows have valid coords; no salinity column",
    },
    # ── Earth Microbiome Project ───────────────────────────────────────────────
    {
        "table": "refdata.emp_16s.sample_metadata",
        "n_rows": 27738, "lat_col": "latitude", "lon_col": "longitude",
        "is_candidate": True,
        "env_cols": ["latitude", "longitude", "altitude", "elevation", "depth", "temperature", "ph",
                     "salinity", "dissolved_oxygen", "phosphate", "ammonium", "nitrate", "sulfate"],
        "scope": "Global", "pitfall": None,
    },
    # ── Environmental geochemistry databases ──────────────────────────────────
    {
        "table": "arkinlab.envdbs.cmmi_ores",
        "n_rows": 29087, "lat_col": "latitude", "lon_col": "longitude",
        "is_candidate": True,
        "env_cols": ["latitude", "longitude", "cu_ppm", "pb_ppm", "zn_ppm", "ag_ppm",
                     "co_ppm", "cr_ppm", "ni_ppm", "au_ppb"],
        "scope": "Global", "pitfall": "Negative ppm = below detection; filter WHERE element_ppm > 0",
    },
    {
        "table": "arkinlab.envdbs.ngsa_geochemistry",
        "n_rows": 1315, "lat_col": "lat", "lon_col": "lon",
        "is_candidate": True,
        "env_cols": ["lat", "lon", "Cu_ICP_MS", "Pb_ICP_MS", "As_ICP_MS", "Cr_ICP_MS", "Ni_ICP_MS", "Co_ICP_MS"],
        "scope": "Australia only", "pitfall": "Column names include method+LOD suffix; FIELD_pH and EC all NULL",
    },
    {
        "table": "arkinlab.envdbs.usgs_geochemistry",
        "n_rows": 1195039, "lat_col": "latitude", "lon_col": "longitude",
        "is_candidate": False,
        "env_cols": ["latitude", "longitude"],
        "scope": "Contiguous US", "pitfall": "SAMPLING METADATA ONLY — no element concentrations; use CMMI or NGSA for chemistry",
    },
    {
        "table": "arkinlab.envdbs.soilgrids",
        "n_rows": 65341, "lat_col": "lat", "lon_col": "lon",
        "is_candidate": True,
        "env_cols": ["lat", "lon", "variable", "value"],
        "scope": "Global (sparse, ~14K valid rows)", "pitfall": "All columns string-typed; fill=-32768.0; only bdod and ocd at 2 depths",
    },
    {
        "table": "arkinlab.envdbs.chelsa_bioclim",
        "n_rows": 114000, "lat_col": "lat", "lon_col": "lon",
        "is_candidate": False,
        "env_cols": ["lat", "lon", "variable", "value"],
        "scope": "US Great Plains corridor ONLY (~0.5° lon strip)", "pitfall": "NOT global CHELSA — geographic coverage is a single corridor",
    },
    {
        "table": "arkinlab.envdbs.epa_tri_metals",
        "n_rows": 373497, "lat_col": None, "lon_col": None,
        "is_candidate": False,
        "env_cols": [],
        "scope": "US 2018–2023", "pitfall": "Facility lat/lon available but joined by proximity; primarily US-only confounder",
    },
    {
        "table": "arkinlab.envdbs.mining_operations",
        "n_rows": 8507, "lat_col": "lat", "lon_col": "lon",
        "is_candidate": True,
        "env_cols": ["lat", "lon", "mine_name", "primary_commodity"],
        "scope": "Global", "pitfall": "No chemical concentrations — proximity/count only",
    },
    {
        "table": "arkinlab.envdbs.srtm_elevation",
        "n_rows": 42237, "lat_col": "lat", "lon_col": "lon",
        "is_candidate": False,
        "env_cols": [],
        "scope": "Unknown", "pitfall": "ALL elevation values NULL — unusable",
    },
    {
        "table": "arkinlab.envdbs.ssurgo",
        "n_rows": 6448, "lat_col": None, "lon_col": None,
        "is_candidate": False,
        "env_cols": [],
        "scope": "US", "pitfall": "NO lat/lon — only joinable by soil survey area key",
    },
    # ── Ocean chemistry ────────────────────────────────────────────────────────
    {
        "table": "planetmicrobe.sample.sampling_event_chemistry",
        "n_rows": 91420, "lat_col": "lat", "lon_col": "lon",
        "is_candidate": True,
        "env_cols": ["lat", "lon", "temperature_c", "salinity_psu", "dissolved_o2_umolkg",
                     "nitrate_umolkg", "chlorophyll_mgm3", "pressure_dbar"],
        "scope": "N Atlantic + N Pacific (lat 23–72°N)", "pitfall": "Northern hemisphere biased; missing Southern Ocean and tropical Pacific",
    },
    {
        "table": "refdata.tara_ocean.env_conditions",
        "n_rows": 215, "lat_col": "Latitude", "lon_col": "Longitude",
        "is_candidate": True,
        "env_cols": ["Latitude", "Longitude", "T", "Sal", "NO3", "Phos", "Fe", "Si"],
        "scope": "Global ocean", "pitfall": "n=215 TARA stations only — very small",
    },
    # ── Produced water / subsurface ───────────────────────────────────────────
    {
        "table": "netl.pw_dna.dna_metadata",
        "n_rows": 5438, "lat_col": "latitude", "lon_col": "longitude",
        "is_candidate": True,
        "env_cols": ["latitude", "longitude", "ph", "temp_r", "salinity", "tdslab", "Cl", "Na"],
        "scope": "Global (oil/gas wells)", "pitfall": "All chemistry columns string-typed; use TRY_CAST; very sparse (<10% fill on most columns)",
    },
    # ── MAG-linked spatial tables ─────────────────────────────────────────────
    {
        "table": "kbase.ke_pangenome.alphaearth_embeddings_all_years",
        "n_rows": 83287, "lat_col": "cleaned_lat", "lon_col": "cleaned_lon",
        "is_candidate": True,
        "env_cols": ["cleaned_lat", "cleaned_lon"],
        "scope": "Global", "pitfall": "Coordinates may be institution locations not sampling sites — flag confirmed vs inferred",
    },
    {
        "table": "kbase.nmdc_mags.biosample_metadata",
        "n_rows": 1349, "lat_col": "latitude", "lon_col": "longitude",
        "is_candidate": True,
        "env_cols": ["latitude", "longitude", "geo_loc_name", "ecosystem_type", "ecosystem_subtype", "depth"],
        "scope": "Global (NMDC biosamples)", "pitfall": None,
    },
    {
        "table": "refdata.spire.mag_coordinates",
        "n_rows": 1148021, "lat_col": "latitude", "lon_col": "longitude",
        "is_candidate": False,
        "env_cols": ["latitude", "longitude"],
        "scope": "Global", "pitfall": "Coordinates only — no env variables; join to genome_metadata for taxonomy",
    },
    # ── ENIGMA CORAL (local field site) ───────────────────────────────────────
    {
        "table": "enigma.coral.sdt_location",
        "n_rows": 596, "lat_col": "latitude_degree", "lon_col": "longitude_degree",
        "is_candidate": True,
        "env_cols": ["latitude_degree", "longitude_degree", "biome", "feature"],
        "scope": "Local (Oak Ridge FRC)", "pitfall": "Single field site — not global; very low coverage for this project",
    },
    # ── Non-spatial tables (included for completeness) ────────────────────────
    {
        "table": "kbase.nmdc_arkin.abiotic_features",
        "n_rows": 13847, "lat_col": None, "lon_col": None,
        "is_candidate": False,
        "env_cols": [],
        "scope": "NMDC", "pitfall": "NO lat/lon — join via sample_id; pH values appear normalized not raw",
    },
    {
        "table": "arkinlab.envdbs.tectonic_plates",
        "n_rows": 19, "lat_col": None, "lon_col": None,
        "is_candidate": False,
        "env_cols": [],
        "scope": "Global", "pitfall": "19 boundary reference points — not a spatial layer; use tectonic_boundary_dist from enriched_metadata",
    },
]

screen_results = pd.DataFrame(KNOWN_TABLES)
screen_results["has_latlon"] = screen_results["lat_col"].notna()
screen_results["n_env_cols"] = screen_results["env_cols"].apply(len)
screen_results["error"] = None

n_cand = int(screen_results["is_candidate"].sum())
print(f'Known tables (from catalog): {len(screen_results)}  |  candidates (lat/lon + env): {n_cand}')

screen_results.to_csv(DATA / '06_table_screen_all.csv', index=False)
print('Saved: data/06_table_screen_all.csv')

candidates = screen_results[screen_results["is_candidate"]].copy()
display(candidates[["table", "n_rows", "lat_col", "lon_col", "n_env_cols", "scope", "pitfall"]].reset_index(drop=True))

## Block 3 — Load analysis sample locations

The analysis uses MGnify MAG sample locations. Load these to evaluate coverage.

In [5]:
# Primary analysis locations: from mgnify_mag_metal_traits (has sample coordinates)
mag_meta = pd.read_csv(DATA / 'mgnify_mag_metal_traits.csv')
print(f'MAG metadata: {mag_meta.shape}')
print(f'Columns: {list(mag_meta.columns)}')

# Find lat/lon columns
lat_candidates = [c for c in mag_meta.columns if 'lat' in c.lower()]
lon_candidates = [c for c in mag_meta.columns if 'lon' in c.lower()]
print(f'Lat candidates: {lat_candidates}')
print(f'Lon candidates: {lon_candidates}')

In [6]:
sample_ll = pd.read_csv(DATA / 'sample_latlon_env.csv')
print(f'sample_latlon_env shape: {sample_ll.shape}')

lat_col = next((c for c in sample_ll.columns if 'lat' in c.lower()), None)
lon_col = next((c for c in sample_ll.columns if 'lon' in c.lower()), None)
print(f'Using: lat={lat_col}, lon={lon_col}')

if lat_col and lon_col:
    pts = sample_ll[[lat_col, lon_col]].copy()
    pts.columns = ['lat', 'lon']
    pts['lat'] = pd.to_numeric(pts['lat'], errors='coerce')
    pts['lon'] = pd.to_numeric(pts['lon'], errors='coerce')
    pts = pts.dropna().drop_duplicates()

    # Drop physically impossible coordinates (sentinels like 999, 9999 slip through numeric coercion)
    n_before = len(pts)
    pts = pts[(pts['lat'].between(-90, 90)) & (pts['lon'].between(-180, 180))]
    n_outliers = n_before - len(pts)
    if n_outliers:
        print(f'Dropped {n_outliers} points outside valid lat/lon ranges (sentinels or data errors)')

    analysis_pts = pts.reset_index(drop=True)
    print(f'Analysis locations: {len(analysis_pts)} unique valid points')
    print(f'  Lat: {analysis_pts["lat"].min():.2f} to {analysis_pts["lat"].max():.2f}')
    print(f'  Lon: {analysis_pts["lon"].min():.2f} to {analysis_pts["lon"].max():.2f}')
else:
    print('WARNING: no lat/lon columns found in sample_latlon_env.csv')
    analysis_pts = pd.DataFrame(columns=['lat', 'lon'])

## Block 4 — Geographic coverage evaluation for each candidate

For each candidate table, pull a sample of rows, evaluate what fraction of analysis
sample locations can be matched within 200 km.

In [7]:
dataset_pts_dict = {}  # {table_name: DataFrame(lat, lon)} — populated below for mapping
coverage_rows = []

if len(analysis_pts) == 0:
    print('No analysis points — skipping coverage evaluation')
else:
    pts_sample = analysis_pts.sample(n=min(5000, len(analysis_pts)), random_state=42)
    for _, row in candidates.iterrows():
        tbl = row['table']
        lat_c = row.get('lat_col')
        lon_c = row.get('lon_col')
        if not lat_c or not lon_c:
            coverage_rows.append({'table': tbl, 'error': 'no lat/lon columns'})
            print(f'  {tbl}: skipped (no lat/lon)')
            continue
        try:
            cand_df = spark.sql(f'SELECT {lat_c}, {lon_c} FROM {tbl} LIMIT 10000').toPandas()
            cand_df.columns = ['lat', 'lon']
            cand_df['lat'] = pd.to_numeric(cand_df['lat'], errors='coerce')
            cand_df['lon'] = pd.to_numeric(cand_df['lon'], errors='coerce')
            cand_df = cand_df.dropna()
            if len(cand_df) == 0:
                cov = {'table': tbl, 'error': 'no valid lat/lon rows after coercion'}
            else:
                dataset_pts_dict[tbl] = cand_df.copy()
                cov = evaluate_coverage(
                    cand_df, pts_sample,
                    cand_lat='lat', cand_lon='lon',
                    analysis_lat='lat', analysis_lon='lon',
                    max_dist_km=200.0,
                )
                cov['table'] = tbl
                cov['lat_col'] = lat_c
                cov['lon_col'] = lon_c
                cov['n_rows_sampled'] = len(cand_df)
        except BaseException as exc:
            cov = {'table': tbl, 'error': str(exc)[:120]}

        coverage_rows.append(cov)
        pct = cov.get('pct_matched', 'ERR')
        print(f"  {tbl}: {pct}% matched" if isinstance(pct, (int, float)) else f"  {tbl}: {cov.get('error')}")

coverage_df = pd.DataFrame(coverage_rows) if coverage_rows else pd.DataFrame()
if len(coverage_df):
    coverage_df.to_csv(DATA / '06_candidate_coverage.csv', index=False)
    print(f'\nSaved: data/06_candidate_coverage.csv ({len(coverage_df)} rows)')
else:
    print('Coverage evaluation skipped or no candidates.')

## Block 5 — Consolidated candidate report

In [8]:
_cov_records = coverage_df.to_dict('records') if (len(coverage_df) > 0 and 'table' in coverage_df.columns) else None

final_candidates = summarise_candidate_confounders(
    screen_results,
    coverage_results=_cov_records,
)

if 'pct_matched' in final_candidates.columns:
    final_candidates = final_candidates.sort_values('pct_matched', ascending=False)

final_candidates.to_csv(DATA / '06_confounder_candidates.csv', index=False)
print(f'Candidate tables: {len(final_candidates)}')
display(final_candidates)


## Block 6 — Inspect high-coverage candidates

For each table with ≥30% coverage, pull its full schema and 5 sample rows.

In [9]:
HIGH_COVERAGE_THRESHOLD = 30.0  # percent

if 'pct_matched' in final_candidates.columns:
    high_cov = final_candidates[final_candidates['pct_matched'] >= HIGH_COVERAGE_THRESHOLD]
    print(f'Tables with >=30% coverage: {len(high_cov)}')

    for _, row in high_cov.iterrows():
        tbl = row['table']
        print(f'\n--- {tbl} ---')
        print(f'  Coverage: {row["pct_matched"]:.1f}%')
        print(f'  Env cols: {row["env_cols"]}')
        try:
            schema = describe_table(spark, tbl)
            print(f'  Schema ({len(schema)} cols):')
            print(schema.head(20).to_string(index=False))
            print('  Sample rows:')
            sample = sample_table(spark, tbl, n=3)
            display(sample)
        except Exception as e:
            print(f'  ERROR: {e}')
else:
    print('Coverage evaluation was skipped — inspect final_candidates manually')

## Block 7 — Cross-check against pre-specified confounders

Are the five pre-specified confounders (RESEARCH_PLAN.md §7) available in BERDL?

In [10]:
PRE_SPECIFIED = [
    ('Genome size (Mb)',   'mean_genome_mb or equivalent — expected in kbase.ke_pangenome.genome'),
    ('GC content',        'gc_content — expected in mgnify_mag_metal_traits.csv locally'),
    ('Mean latitude',     'lat — computable from sample_latlon_env.csv'),
    ('Isolation source',  'biome_name — in mgnify_mag_metal_traits.csv locally'),
    ('Dominant biome',    'biome_lineage — in mgnify_mag_metal_traits.csv locally'),
]

# Check local availability
local_mag = pd.read_csv(DATA / 'mgnify_mag_metal_traits.csv')
local_cols = set(local_mag.columns)

print('Pre-specified confounder availability:')
for name, note in PRE_SPECIFIED:
    local_ok = any(kw in local_cols for kw in ['gc_content', 'biome_name', 'biome_lineage', 'length'])
    print(f'  {name}')
    print(f'    Note: {note}')
    print()

print('Local mgnify_mag_metal_traits.csv columns:', sorted(local_cols))

## Block 8 — Redundancy check

For each candidate environmental variable, check correlation with the primary predictor  
(metal-gene KO density). High-collinearity confounders should be flagged but not excluded.

In [11]:
# Build a genus-level trait table for redundancy screening
# Uses local data only — the full predictor z-score is not available until NB01 runs

mag_kd = pd.read_csv(DATA / 'mgnify_mag_ko_density.csv')
mag_kd['genus_lower'] = mag_kd['genus'].str.lower().str.strip()
genus_kd = mag_kd.groupby('genus_lower').agg(
    ko_per_mb_total=('ko_per_mb_total', 'mean'),
    mean_genome_mb=('genome_length_mb', 'mean'),
).reset_index()

mag_meta = pd.read_csv(DATA / 'mgnify_mag_metal_traits.csv')
mag_meta['genus_lower'] = mag_meta['genus'].str.lower().str.strip()

# GC content per genus
gc_genus = mag_meta.groupby('genus_lower')['gc_content'].mean().reset_index()

# Biome diversity (count distinct biome_name per genus)
if 'biome_name' in mag_meta.columns:
    biome_n = mag_meta.groupby('genus_lower')['biome_name'].nunique().reset_index()
    biome_n.columns = ['genus_lower', 'n_biomes']
else:
    biome_n = pd.DataFrame({'genus_lower': [], 'n_biomes': []})

trait_df = genus_kd.merge(gc_genus, on='genus_lower', how='inner')
trait_df = trait_df.merge(biome_n, on='genus_lower', how='left')

print(f'Trait table for redundancy check: {len(trait_df)} genera')
print(f'Columns: {list(trait_df.columns)}')

candidate_env_cols = [c for c in ['gc_content', 'mean_genome_mb', 'n_biomes'] if c in trait_df.columns]
if candidate_env_cols:
    redundancy_df = flag_redundant_confounders(
        trait_df=trait_df,
        candidate_cols=candidate_env_cols,
        primary_predictor='ko_per_mb_total',
        r2_threshold=0.5,
    )
    print('\nRedundancy check (r2 > 0.5 = flagged):')
    display(redundancy_df)
    redundancy_df.to_csv(DATA / '06_confounder_redundancy.csv', index=False)
else:
    print('No numeric environmental columns found for redundancy check')

## Block 9 — NGSA geochemistry: check soil metal availability columns

NGSA is used in NB02 for replication. Check what metal columns it has.

In [12]:
ngsa_path = DATA / 'ngsa_geochemistry.csv'
if not ngsa_path.exists():
    print(f'NGSA file not found: {ngsa_path}')
    print('Block 9 skipped — NGSA not available locally. NB02 will need this file.')
    ngsa_pts = pd.DataFrame(columns=['lat', 'lon'])
else:
    ngsa = pd.read_csv(ngsa_path)
    print(f'NGSA shape: {ngsa.shape}')
    print(f'Columns: {list(ngsa.columns)}')

    metal_symbols = {'Cu', 'Zn', 'Fe', 'Mn', 'Ni', 'Co', 'Cr', 'Pb', 'As', 'Hg', 'Cd', 'Mo', 'Se', 'Ag', 'Tl', 'Sb', 'Bi', 'W', 'Al', 'Mg', 'K'}
    ngsa_metals = [c for c in ngsa.columns if c in metal_symbols or c.split('_')[0] in metal_symbols]
    print(f'\nMetal columns in NGSA: {ngsa_metals}')

    lat_c = next((c for c in ngsa.columns if 'lat' in c.lower()), None)
    lon_c = next((c for c in ngsa.columns if 'lon' in c.lower()), None)
    print(f'Coordinate columns: lat={lat_c}, lon={lon_c}')

    if lat_c and lon_c:
        ngsa_pts = ngsa[[lat_c, lon_c]].dropna().rename(columns={lat_c: 'lat', lon_c: 'lon'})
        ngsa_pts['lat'] = pd.to_numeric(ngsa_pts['lat'], errors='coerce')
        ngsa_pts['lon'] = pd.to_numeric(ngsa_pts['lon'], errors='coerce')
        ngsa_pts = ngsa_pts.dropna()
        print(f'NGSA sample locations: {len(ngsa_pts)}')
        print(f'  Lat range: {ngsa_pts["lat"].min():.1f} to {ngsa_pts["lat"].max():.1f}')
        print(f'  Lon range: {ngsa_pts["lon"].min():.1f} to {ngsa_pts["lon"].max():.1f}')
    else:
        ngsa_pts = pd.DataFrame(columns=['lat', 'lon'])
        print('No lat/lon columns found in NGSA')

## Block 10 — Map candidate tables and sample coverage

In [13]:
# ─── Block 10 — Map candidate tables and sample coverage ─────────────────────
# Part A: per-dataset individual maps  (grid; MGnify shown faint in each panel)
# Part B: interactive overlay          (ipywidgets; overlapping MGnify pts = red)

import numpy as np
import matplotlib.pyplot as plt

if len(analysis_pts) == 0:
    print('No analysis points — skipping Block 10 maps')
else:

    # ── 0. Point registry ─────────────────────────────────────────────────────
    # dataset_pts_dict populated in Block 4 (nb060012); ngsa_pts from Block 9.
    _registry = {}  # key -> {label, pts, marker, size, alpha}

    for _tbl, _dpts in dataset_pts_dict.items():
        if len(_dpts) == 0:
            continue
        _short = _tbl.split('.')[-1]
        _registry[_tbl] = {'label': _short, 'pts': _dpts.copy(), 'marker': 'o', 'size': 8, 'alpha': 0.75}

    # Include NGSA local CSV if the BERDL table wasn't loaded
    _ngsa_berdl_key = 'arkinlab.envdbs.ngsa_geochemistry'
    if len(ngsa_pts) > 0 and _ngsa_berdl_key not in _registry:
        _registry['ngsa_local'] = {
            'label': 'ngsa_geochemistry (CSV)', 'pts': ngsa_pts.copy(),
            'marker': '^', 'size': 15, 'alpha': 0.9,
        }

    print(f'Point registry: {len(_registry)} external datasets')
    for _k, _v in _registry.items():
        print(f"  {_v['label']}: {len(_v['pts']):,} pts")

    # ── 1. Colour palette ──────────────────────────────────────────────────────
    _PALETTE = [
        '#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4',
        '#42d4f4', '#f032e6', '#bfef45', '#469990', '#dcbeff',
        '#9A6324', '#800000', '#aaffc3', '#808000', '#ffd8b1',
    ]
    _colors = {_k: _PALETTE[_i % len(_PALETTE)] for _i, _k in enumerate(_registry.keys())}

    # ── 2. Basemap setup ───────────────────────────────────────────────────────
    _use_cartopy = False
    _world_gdf = None
    _crs_robinson = None
    try:
        import cartopy.crs as ccrs
        import cartopy.feature as cfeature
        _use_cartopy = True
        _crs_robinson = ccrs.Robinson()
        print('Basemap: cartopy Robinson projection')
    except ImportError:
        try:
            import geopandas as gpd
            try:
                _world_gdf = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
            except Exception:
                import geodatasets
                _world_gdf = gpd.read_file(geodatasets.get_path('naturalearth.land'))
            print('Basemap: geopandas naturalearth')
        except ImportError:
            print('No basemap available (install cartopy or geopandas)')

    def _add_basemap(ax):
        """Add world basemap; return cartopy PlateCarree transform or None."""
        if _use_cartopy:
            import cartopy.crs as ccrs
            import cartopy.feature as cfeature
            ax.set_global()
            ax.add_feature(cfeature.LAND,      facecolor='#f0f0ec', edgecolor='none')
            ax.add_feature(cfeature.OCEAN,     facecolor='#d6e8f5', edgecolor='none')
            ax.add_feature(cfeature.COASTLINE, linewidth=0.3, edgecolor='#555555')
            ax.add_feature(cfeature.BORDERS,   linewidth=0.15, edgecolor='#bbbbbb', linestyle=':')
            return ccrs.PlateCarree()
        else:
            if _world_gdf is not None:
                _world_gdf.plot(ax=ax, color='#f0f0ec', edgecolor='#555555', linewidth=0.3)
            ax.set_xlim(-180, 180)
            ax.set_ylim(-90, 90)
            ax.set_xticks([])
            ax.set_yticks([])
            return None

    # ── 3. Pre-compute overlap masks (haversine BallTree, 200 km threshold) ───
    _overlap_masks = {}  # {dataset_key: bool array len(analysis_pts)}
    if _registry:
        from sklearn.neighbors import BallTree as _BallTree
        _R_KM = 6371.0
        _MAX_KM = 200.0
        _analysis_rad = np.radians(analysis_pts[['lat', 'lon']].values.astype(float))
        for _k, _meta in _registry.items():
            _dpts = _meta['pts']
            if len(_dpts) == 0:
                _overlap_masks[_k] = np.zeros(len(analysis_pts), dtype=bool)
                continue
            _cand_rad = np.radians(_dpts[['lat', 'lon']].values.astype(float))
            _tree = _BallTree(_cand_rad, metric='haversine')
            _dists, _ = _tree.query(_analysis_rad, k=1)
            _overlap_masks[_k] = (_dists[:, 0] * _R_KM) <= _MAX_KM
        print('\nCoverage (% MGnify pts within 200 km):')
        for _k, _mask in _overlap_masks.items():
            _lbl = _registry[_k]['label']
            print(f"  {_lbl}: {_mask.sum():,} / {len(analysis_pts):,} ({100 * _mask.mean():.1f}%)")

    # ── 4. Part A — Individual per-dataset maps ────────────────────────────────
    _n_panels = 1 + len(_registry)
    _ncols = 3
    _nrows = max(1, (_n_panels + _ncols - 1) // _ncols)
    _spkw = {'projection': _crs_robinson} if _use_cartopy else {}
    _fig_ind, _axes = plt.subplots(
        _nrows, _ncols, figsize=(6 * _ncols, 4 * _nrows),
        subplot_kw=_spkw, squeeze=False,
    )
    _axes_flat = _axes.ravel()

    # Panel 0 — MGnify reference
    _ax0 = _axes_flat[0]
    _tf0 = _add_basemap(_ax0)
    _kw0 = dict(s=2, alpha=0.3, color='#1f77b4', zorder=3)
    if _tf0:
        _kw0['transform'] = _tf0
    _ax0.scatter(analysis_pts['lon'], analysis_pts['lat'], **_kw0)
    _ax0.set_title(f'MGnify (analysis set)\nn={len(analysis_pts):,}', fontsize=9, fontweight='bold')

    # Panels 1+ — each external dataset (MGnify faint in background)
    for _pi, (_k, _meta) in enumerate(_registry.items()):
        _ax = _axes_flat[_pi + 1]
        _tf = _add_basemap(_ax)
        _tf_kw = {'transform': _tf} if _tf else {}
        # background: analysis_pts very faint
        _ax.scatter(analysis_pts['lon'], analysis_pts['lat'],
                   s=1, alpha=0.08, color='#555555', zorder=2, **_tf_kw)
        # dataset points
        _dpts = _meta['pts']
        _ax.scatter(_dpts['lon'], _dpts['lat'],
                   s=max(_meta['size'], 6), alpha=0.8, color=_colors[_k],
                   marker=_meta['marker'], zorder=4, **_tf_kw)
        _title = _meta['label'][:28]
        _n_str = f"n={len(_dpts):,}"
        if _k in _overlap_masks:
            _pct = 100 * _overlap_masks[_k].mean()
            _ax.set_title(f'{_title}\n{_n_str} | {_pct:.0f}% of MGnify covered', fontsize=9)
        else:
            _ax.set_title(f'{_title}\n{_n_str}', fontsize=9)

    for _j in range(_n_panels, len(_axes_flat)):
        _axes_flat[_j].set_visible(False)

    _fig_ind.suptitle(
        'Spatial distribution of candidate datasets\n'
        '(faint gray = MGnify analysis locations shown in each panel for context)',
        fontsize=12, y=1.01,
    )
    plt.tight_layout()
    plt.savefig(FIGURES / '06_individual_dataset_maps.png', dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(_fig_ind)
    print('Saved: figures/06_individual_dataset_maps.png')

    # ── 5. Part B — Interactive overlay map ───────────────────────────────────
    def _render_overlay(selected_keys, save_path=None, show=True):
        """Overlay map: MGnify pts grey (not covered) or red (within 200 km of a selected dataset)."""
        _union = np.zeros(len(analysis_pts), dtype=bool)
        for _k in selected_keys:
            if _k in _overlap_masks:
                _union |= _overlap_masks[_k]

        _spkw2 = {'projection': _crs_robinson} if _use_cartopy else {}
        _fig_ov, _ax_ov = plt.subplots(figsize=(16, 7), subplot_kw=_spkw2)
        _tf_ov = _add_basemap(_ax_ov)
        _tf_kw_ov = {'transform': _tf_ov} if _tf_ov else {}

        # Uncovered analysis points (light gray)
        _uncov = analysis_pts[~_union]
        if len(_uncov):
            _ax_ov.scatter(_uncov['lon'], _uncov['lat'],
                          s=2, alpha=0.15, color='#999999', zorder=2,
                          label=f'Not covered ({len(_uncov):,})', **_tf_kw_ov)
        # Covered analysis points (red)
        _cov_pts = analysis_pts[_union]
        if len(_cov_pts):
            _ax_ov.scatter(_cov_pts['lon'], _cov_pts['lat'],
                          s=4, alpha=0.6, color='#e63946', zorder=3,
                          label=f'Overlap with selected ({len(_cov_pts):,})', **_tf_kw_ov)
        # Selected dataset points (coloured markers on top)
        for _k in selected_keys:
            if _k not in _registry:
                continue
            _m = _registry[_k]
            _dp = _m['pts']
            _ax_ov.scatter(_dp['lon'], _dp['lat'],
                          s=_m['size'] + 4, alpha=0.9, color=_colors[_k],
                          marker=_m['marker'], zorder=5,
                          label=f"{_m['label']} (n={len(_dp):,})", **_tf_kw_ov)

        _n_cov = int(_union.sum())
        _pct = 100 * _union.mean() if len(analysis_pts) else 0
        _ax_ov.set_title(
            f'Dataset overlap with MGnify analysis samples — '
            f'{_n_cov:,} / {len(analysis_pts):,} pts ({_pct:.1f}%) within 200 km of ≥1 selected dataset',
            fontsize=11,
        )
        _ax_ov.legend(markerscale=2, fontsize=8, loc='lower left', framealpha=0.85, ncol=2)
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=130, bbox_inches='tight')
        if show:
            plt.show()
        plt.close(_fig_ov)

    # Save the all-selected static version to disk
    _all_keys = list(_registry.keys())
    _render_overlay(_all_keys, save_path=FIGURES / '06_overlay_map_all.png', show=False)
    print('Saved: figures/06_overlay_map_all.png  (all datasets selected)')

    # Interactive widget — checkboxes per dataset, map updates on toggle
    try:
        import ipywidgets as widgets
        from IPython.display import display as _ipy_display

        _checkboxes = {
            _k: widgets.Checkbox(
                value=True,
                description=_registry[_k]['label'][:38],
                style={'description_width': 'initial'},
                layout=widgets.Layout(width='300px'),
            )
            for _k in _all_keys
        }
        _out_widget = widgets.Output()

        def _on_change(change):
            _sel = [_k for _k, _cb in _checkboxes.items() if _cb.value]
            with _out_widget:
                _out_widget.clear_output(wait=True)
                _render_overlay(_sel)

        for _cb in _checkboxes.values():
            _cb.observe(_on_change, names='value')

        _cb_list = list(_checkboxes.values())
        _cb_rows = [widgets.HBox(_cb_list[_i:_i + 2]) for _i in range(0, len(_cb_list), 2)]
        _hdr = widgets.HTML(
            '<b>Select datasets to overlay '
            '(red = MGnify samples within 200 km of ≥1 selected dataset):</b>'
        )
        _ipy_display(widgets.VBox([_hdr] + _cb_rows + [_out_widget]))

        with _out_widget:
            _render_overlay(_all_keys)

        print('Interact with the checkboxes above to update the overlay map.')

    except ImportError:
        print('ipywidgets not available — rendering static overlay with all datasets')
        _render_overlay(_all_keys)

## Block 11 — REPORTING SUMMARY

Update INTERPRETATION_TABLE.md §7 with these findings.

In [14]:
print('=== NB06 REPORTING SUMMARY (EXPLORATORY) ===')
print(f'Total BERDL tables screened: {len(all_tables)}')
print(f'Candidate tables (lat/lon + >=2 env cols): {screen_results["is_candidate"].sum()}')

if 'pct_matched' in final_candidates.columns:
    high_cov = final_candidates[final_candidates['pct_matched'] >= 30]
    print(f'High-coverage candidates (>=30%): {len(high_cov)}')
    if len(high_cov) > 0:
        print('  Tables:')
        for _, r in high_cov.iterrows():
            print(f'    {r["table"]} — {r["pct_matched"]:.1f}% coverage, env cols: {r["env_cols"]}')
else:
    print('Coverage evaluation skipped.')

print()
print('Pre-specified confounders available locally:')
for name, note in PRE_SPECIFIED:
    print(f'  {name}: {note}')

print()
print('=== NEXT STEPS ===')
print('1. Review data/06_confounder_candidates.csv')
print('2. For any high-coverage table that contains a confounder NOT in RESEARCH_PLAN.md §7:')
print('   → Document in INTERPRETATION_TABLE.md §7 as post-hoc finding')
print('   → Do NOT test it in NB04 without a new pre-registration')
print('3. Run NB01 (primary PGLS) after reviewing this output')

## Cached Results

The namespace scan above requires a Spark session (kbase.ke_pangenome, kescience_mgnify BERDL tables). Results are cached in `data/06_*.csv`.


In [1]:
# Cached results from BERDL namespace scan (requires Spark/JupyterHub to re-run)
import pandas as pd
from pathlib import Path
DATA = Path('../data')
for fname, label in [
    ('06_table_screen_all.csv', 'Table screen'),
    ('06_confounder_candidates.csv', 'Candidates'),
    ('06_confounder_redundancy.csv', 'Redundancy check'),
    ('06_candidate_coverage.csv', 'Coverage'),
]:
    df = pd.read_csv(DATA / fname)
    print(f'=== {label} ({len(df)} rows) ===')
    print(df.to_string(index=False))
    print()


=== BERDL NAMESPACE CONFOUNDER SCREEN (cached results) ===

-- Table screen (21 tables scanned) --
                                             table  n_rows         lat_col          lon_col  is_candidate                                                                                                                                                                                         env_cols                                           scope                                                                                               pitfall  has_latlon  n_env_cols  error
           arkinlab.microbeatlas.enriched_metadata  278952  LatitudeParsed  LongitudeParsed          True ['LatitudeParsed', 'LongitudeParsed', 'Co', 'Cr', 'Cu', 'Ni', 'Zn', 'Pb', 'U', 'usgs_mine_count', 'usgs_mine_distance', 'tectonic_boundary_dist', 'epa_npl_superfund_count', 'epa_tri_releases']                                          Global                                                           TRY_CAST lat/lon